# 🔥 Interactive Wildfire Burn Severity Visualization
## Using `ipyleaflet` + Sentinel-2 Imagery

**Course:** Advanced GEP Techniques  
**Package Demo:** `ipyleaflet`

---

### What this notebook does:
1. Loads Sentinel-2 GeoTIFF bands (pre-fire and post-fire)
2. Computes **NBR** (Normalized Burn Ratio) and **dNBR** (differenced NBR)
3. Classifies burn severity into 5 categories
4. Exports composites as web-viewable images
5. Visualizes everything interactively with **ipyleaflet**

## ⚙️ Step 0 — Install Dependencies

In [ ]:
# Run this cell once — Colab will restart if needed
!pip install -q ipyleaflet rasterio matplotlib numpy pillow ipywidgets
print("✅ All packages installed")

## 📦 Step 1 — Imports & Configuration

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.cm as cm
import io, base64, os, warnings
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import (
    Map, TileLayer, ImageOverlay, LayersControl,
    WidgetControl, basemaps, basemap_to_tiles, ScaleControl
)

warnings.filterwarnings('ignore')
print("✅ Imports complete")

## 📂 Step 2 — Upload Your Sentinel-2 GeoTIFF Files

Upload the following files using the **Files panel** on the left (or the cell below):

| Variable | Description | Sentinel-2 Bands |
|----------|-------------|-------------------|
| `pre_nir` | Pre-fire NIR band | B08 |
| `pre_swir` | Pre-fire SWIR band | B12 |
| `pre_rgb` | Pre-fire RGB composite | B04, B03, B02 |
| `post_nir` | Post-fire NIR band | B08 |
| `post_swir` | Post-fire SWIR band | B12 |
| `post_rgb` | Post-fire RGB composite | B04, B03, B02 |

> 💡 **Tip:** If you have a single multi-band GeoTIFF, set the band numbers in the config cell below.

In [ ]:
from google.colab import files

print("Select your GeoTIFF files to upload...")
uploaded = files.upload()

print("\n📁 Uploaded files:")
for fname in uploaded:
    size_kb = len(uploaded[fname]) / 1024
    print(f"  {fname}  ({size_kb:.1f} KB)")

## 🔧 Step 3 — Configure File Paths

**Edit the paths below to match your uploaded filenames.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURE THESE PATHS — update to match your uploaded filenames
# ─────────────────────────────────────────────────────────────────────────────

# Option A: Separate single-band GeoTIFFs
CONFIG = {
    "pre_nir":  "pre_fire_B08.tif",     # Pre-fire NIR (B8)
    "pre_swir": "pre_fire_B12.tif",     # Pre-fire SWIR (B12)
    "pre_rgb":  "pre_fire_RGB.tif",     # Pre-fire RGB (3-band or single R band)
    "post_nir": "post_fire_B08.tif",    # Post-fire NIR (B8)
    "post_swir":"post_fire_B12.tif",    # Post-fire SWIR (B12)
    "post_rgb": "post_fire_RGB.tif",    # Post-fire RGB
}

# Option B: If using multi-band GeoTIFF, uncomment and set band numbers:
# MULTI_BAND_FILE = "sentinel2_stack.tif"
# BAND_NIR = 4   # Band index (1-based) for NIR
# BAND_SWIR = 6  # Band index for SWIR
# BANDS_RGB = (3, 2, 1)  # (R, G, B) band indices

# Map display center — update to your study area!
MAP_CENTER = (42.13, -115.34)  # (lat, lon) — McDermitt/Oregon fire area
MAP_ZOOM = 10

print("✅ Configuration set")

# Verify files exist
all_ok = True
for key, path in CONFIG.items():
    if os.path.exists(path):
        print(f"  ✅  {key}: {path}")
    else:
        print(f"  ❌  {key}: {path}  ← NOT FOUND")
        all_ok = False

if not all_ok:
    print("\n⚠️  Some files missing — update paths above and re-run")
else:
    print("\n🚀  All files found — ready to process!")

## 🔢 Step 4 — Compute NBR and dNBR

$$\text{NBR} = \frac{\text{NIR} - \text{SWIR}}{\text{NIR} + \text{SWIR}}$$

$$\text{dNBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$$

Higher dNBR → more severe burn

In [ ]:
def read_band(path, band=1):
    """Read a single band from a GeoTIFF, reproject to WGS84 if needed."""
    with rasterio.open(path) as src:
        # Reproject to WGS84 for leaflet compatibility
        if src.crs != CRS.from_epsg(4326):
            transform, width, height = calculate_default_transform(
                src.crs, CRS.from_epsg(4326), src.width, src.height, *src.bounds
            )
            data = np.zeros((height, width), dtype=np.float32)
            reproject(
                source=rasterio.band(src, band),
                destination=data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=CRS.from_epsg(4326),
                resampling=Resampling.bilinear
            )
            meta = src.meta.copy()
            meta.update({'crs': CRS.from_epsg(4326), 'transform': transform,
                         'width': width, 'height': height})
        else:
            data = src.read(band).astype(np.float32)
            meta = src.meta.copy()
        return data, meta

def get_bounds_wgs84(path):
    """Return [[south, west], [north, east]] in WGS84 for leaflet."""
    with rasterio.open(path) as src:
        if src.crs != CRS.from_epsg(4326):
            from rasterio.warp import transform_bounds
            left, bottom, right, top = transform_bounds(
                src.crs, CRS.from_epsg(4326), *src.bounds
            )
        else:
            left, bottom, right, top = src.bounds
    return [[bottom, left], [top, right]]

def compute_nbr(nir, swir):
    """Compute Normalized Burn Ratio."""
    nir = nir.astype(np.float32)
    swir = swir.astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        nbr = np.where(
            (nir + swir) != 0,
            (nir - swir) / (nir + swir),
            np.nan
        )
    return nbr.clip(-1, 1)

print("📖 Reading bands...")
pre_nir_data,  meta = read_band(CONFIG['pre_nir'])
pre_swir_data, _    = read_band(CONFIG['pre_swir'])
post_nir_data, _    = read_band(CONFIG['post_nir'])
post_swir_data,_    = read_band(CONFIG['post_swir'])

print("🔢 Computing NBR...")
nbr_pre  = compute_nbr(pre_nir_data, pre_swir_data)
nbr_post = compute_nbr(post_nir_data, post_swir_data)
dnbr     = nbr_pre - nbr_post

# Get spatial bounds from NIR file (all bands share same extent)
bounds = get_bounds_wgs84(CONFIG['pre_nir'])

print(f"\n📊 dNBR statistics:")
print(f"  Min:  {np.nanmin(dnbr):.3f}")
print(f"  Max:  {np.nanmax(dnbr):.3f}")
print(f"  Mean: {np.nanmean(dnbr):.3f}")
print(f"  Shape: {dnbr.shape}")
print(f"\n🌍 Bounds (WGS84): {bounds}")
print("\n✅ NBR and dNBR computed successfully")

## 🗂️ Step 5 — Classify Burn Severity

Using USGS dNBR thresholds:

| Class | dNBR Range | Color |
|-------|-----------|-------|
| Enhanced Regrowth | < −0.10 | Dark Green |
| Unburned | −0.10 – 0.10 | Light Green |
| Low Severity | 0.10 – 0.27 | Yellow |
| Moderate Severity | 0.27 – 0.44 | Orange |
| High Severity | > 0.44 | Dark Red |

In [ ]:
# ── Severity classification ────────────────────────────────────────────────
SEVERITY_CLASSES = [
    {"label": "Enhanced Regrowth", "min": -2.0,  "max": -0.10, "color": "#006400", "rgba": (0, 100, 0, 200)},
    {"label": "Unburned",          "min": -0.10, "max":  0.10, "color": "#90EE90", "rgba": (144, 238, 144, 200)},
    {"label": "Low Severity",      "min":  0.10, "max":  0.27, "color": "#FFD700", "rgba": (255, 215, 0, 200)},
    {"label": "Moderate Severity", "min":  0.27, "max":  0.44, "color": "#FF8C00", "rgba": (255, 140, 0, 200)},
    {"label": "High Severity",     "min":  0.44, "max":  2.0,  "color": "#8B0000", "rgba": (139, 0, 0, 220)},
]

def classify_dnbr(dnbr_arr):
    """Create RGBA classified array from dNBR."""
    h, w = dnbr_arr.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    for cls in SEVERITY_CLASSES:
        mask = (dnbr_arr >= cls["min"]) & (dnbr_arr < cls["max"]) & ~np.isnan(dnbr_arr)
        rgba[mask] = cls["rgba"]
    # Transparent where NaN
    nan_mask = np.isnan(dnbr_arr)
    rgba[nan_mask] = [0, 0, 0, 0]
    return rgba

print("🗂️ Classifying dNBR...")
severity_rgba = classify_dnbr(dnbr)

# Count pixels per class
print("\n📊 Severity distribution:")
for cls in SEVERITY_CLASSES:
    mask = (dnbr >= cls["min"]) & (dnbr < cls["max"]) & ~np.isnan(dnbr)
    pct = 100 * mask.sum() / np.sum(~np.isnan(dnbr))
    print(f"  {cls['label']:25s}  {pct:5.1f}%  ({'█' * int(pct/2)})")

print("\n✅ Classification complete")

## 📤 Step 6 — Export Layers as PNG for ipyleaflet

ipyleaflet's `ImageOverlay` accepts a URL or base64-encoded PNG image.

In [ ]:
def array_to_base64_png(arr_2d, cmap='RdYlGn_r', vmin=-1, vmax=1, alpha=0.75):
    """Convert 2D array to base64 PNG using a colormap."""
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colormap = plt.get_cmap(cmap)
    rgba = colormap(norm(np.nan_to_num(arr_2d, nan=0)))
    rgba[..., 3] = np.where(np.isnan(arr_2d), 0, alpha)  # transparent where NaN
    rgba_uint8 = (rgba * 255).astype(np.uint8)
    img = Image.fromarray(rgba_uint8, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def rgba_array_to_base64_png(rgba_arr):
    """Convert RGBA uint8 array to base64 PNG."""
    img = Image.fromarray(rgba_arr.astype(np.uint8), 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def rgb_tif_to_base64_png(path, alpha=0.9):
    """Load RGB GeoTIFF and convert to base64 PNG."""
    with rasterio.open(path) as src:
        if src.count >= 3:
            r = src.read(1).astype(np.float32)
            g = src.read(2).astype(np.float32)
            b = src.read(3).astype(np.float32)
        else:
            # Single-band fallback — display as grayscale
            gray = src.read(1).astype(np.float32)
            r = g = b = gray

    # Normalize to 0-255 (handle Sentinel-2 reflectance scaling)
    def norm_band(band):
        p2, p98 = np.nanpercentile(band, 2), np.nanpercentile(band, 98)
        return np.clip((band - p2) / (p98 - p2 + 1e-8) * 255, 0, 255).astype(np.uint8)

    rgba = np.stack([norm_band(r), norm_band(g), norm_band(b),
                     np.full_like(r, int(255 * alpha), dtype=np.uint8)], axis=-1)
    img = Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

print("📤 Exporting layers...")
url_nbr_pre  = array_to_base64_png(nbr_pre,  cmap='RdYlGn', vmin=-1, vmax=1, alpha=0.8)
url_nbr_post = array_to_base64_png(nbr_post, cmap='RdYlGn', vmin=-1, vmax=1, alpha=0.8)
url_dnbr     = array_to_base64_png(dnbr,     cmap='RdYlGn_r', vmin=-0.5, vmax=0.8, alpha=0.8)
url_severity = rgba_array_to_base64_png(severity_rgba)
url_pre_rgb  = rgb_tif_to_base64_png(CONFIG['pre_rgb'])
url_post_rgb = rgb_tif_to_base64_png(CONFIG['post_rgb'])

print("  ✅ Pre-fire NBR")
print("  ✅ Post-fire NBR")
print("  ✅ dNBR continuous")
print("  ✅ dNBR classified (severity)")
print("  ✅ Pre-fire RGB composite")
print("  ✅ Post-fire RGB composite")
print("\n✅ All layers exported")

## 🔍 Step 7 — Quick Visual Check (static plot)

Before loading into ipyleaflet, let's verify the outputs look correct.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.patch.set_facecolor('#0D1117')

panels = [
    (nbr_pre,      'RdYlGn',   -1,    1,    'Pre-fire NBR'),
    (nbr_post,     'RdYlGn',   -1,    1,    'Post-fire NBR'),
    (dnbr,         'RdYlGn_r', -0.5,  0.8,  'dNBR'),
    (severity_rgba,None,        None,  None, 'Classified Severity'),
]

for ax, (data, cmap, vmin, vmax, title) in zip(axes, panels):
    ax.set_facecolor('#0D1117')
    ax.set_title(title, color='#E6EDF3', fontsize=12, pad=8)
    ax.axis('off')
    if cmap:
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    else:
        ax.imshow(data)  # RGBA directly

# Legend for severity
legend_patches = [Patch(color=c['color'], label=c['label']) for c in SEVERITY_CLASSES]
axes[3].legend(handles=legend_patches, loc='lower left', fontsize=8,
               facecolor='#161B22', labelcolor='white')

plt.suptitle('Wildfire Burn Severity — Static Preview', color='#00D4FF',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("✅ Static preview complete — layers look good!")

---
# 🗺️ DEMO — Interactive Map with ipyleaflet

Now we build the full interactive visualization. Each step is shown separately so the class can follow along.

## 🗺️ Demo Step 1 — Create the Map Widget

In [ ]:
# ── Step 1: Create the Map ─────────────────────────────────────────────────
# Map() creates an interactive Leaflet widget.
# center = (lat, lon), zoom = initial zoom level

m = Map(
    center=MAP_CENTER,
    zoom=MAP_ZOOM,
    scroll_wheel_zoom=True,   # mouse-wheel zoom
    layout=widgets.Layout(width='100%', height='500px')
)

# Just the bare map — we'll add layers in the next steps
display(m)
print("✅ Map widget created — this is an interactive Leaflet map")

## 🗂️ Demo Step 2 — Add Base Tile Layer

`TileLayer` connects to a tile server that serves map tiles (256×256 pixel images) at different zoom levels. ipyleaflet ships with built-in basemaps.

In [ ]:
# ── Step 2: Add different basemaps ─────────────────────────────────────────
# ipyleaflet has many built-in basemaps.
# We'll add a satellite and a terrain version so users can toggle.

satellite_layer = basemap_to_tiles(basemaps.Esri.WorldImagery)
satellite_layer.name = "🛰️ Satellite (ESRI)"
satellite_layer.base = True

terrain_layer = basemap_to_tiles(basemaps.OpenTopoMap)
terrain_layer.name = "🏔️ Terrain (OpenTopo)"
terrain_layer.base = True
terrain_layer.visible = False  # Start hidden

m.add_layer(satellite_layer)
m.add_layer(terrain_layer)

print("✅ Satellite and terrain base layers added")
print("   Notice: the map now shows satellite imagery — you can pan and zoom")

## 🔥 Demo Step 3 — Overlay Raster Layers

`ImageOverlay` places a georeferenced image (PNG) on the map using lat/lon bounds.

In [ ]:
# ── Step 3: Add raster overlays ─────────────────────────────────────────────
# ImageOverlay(url=..., bounds=[[south, west], [north, east]])

# Pre-fire RGB
layer_pre_rgb = ImageOverlay(
    url=url_pre_rgb,
    bounds=bounds,
    name="📸 Pre-fire RGB",
    opacity=0.9
)

# Post-fire RGB
layer_post_rgb = ImageOverlay(
    url=url_post_rgb,
    bounds=bounds,
    name="🔥 Post-fire RGB",
    opacity=0.9,
    visible=False
)

# NBR Pre-fire (continuous)
layer_nbr_pre = ImageOverlay(
    url=url_nbr_pre,
    bounds=bounds,
    name="📊 NBR Pre-fire",
    opacity=0.8,
    visible=False
)

# NBR Post-fire (continuous)
layer_nbr_post = ImageOverlay(
    url=url_nbr_post,
    bounds=bounds,
    name="📊 NBR Post-fire",
    opacity=0.8,
    visible=False
)

# dNBR continuous
layer_dnbr = ImageOverlay(
    url=url_dnbr,
    bounds=bounds,
    name="📈 dNBR (continuous)",
    opacity=0.85,
    visible=False
)

# Classified severity
layer_severity = ImageOverlay(
    url=url_severity,
    bounds=bounds,
    name="🗺️ Burn Severity (classified)",
    opacity=0.85,
    visible=True
)

# Add all layers
for layer in [layer_pre_rgb, layer_post_rgb, layer_nbr_pre,
              layer_nbr_post, layer_dnbr, layer_severity]:
    m.add_layer(layer)

print("✅ All raster layers added to the map")
print("   Layers added:")
print("     • Pre-fire RGB")
print("     • Post-fire RGB")
print("     • NBR Pre-fire")
print("     • NBR Post-fire")
print("     • dNBR continuous")
print("     • Burn Severity classified (visible by default)")

## 🎛️ Demo Step 4 — Add Layer Control

`LayersControl` adds a toggle panel — users can click to show/hide any layer.

In [ ]:
# ── Step 4: Add LayersControl ───────────────────────────────────────────────
# This adds the familiar toggle panel in the top-right corner.
# Users can now switch between layers interactively!

layers_control = LayersControl(position='topright', collapsed=False)
m.add_control(layers_control)

# Also add a scale bar
scale_control = ScaleControl(position='bottomleft')
m.add_control(scale_control)

print("✅ LayersControl added — top-right panel lets you toggle all layers")
print("✅ ScaleControl added — bottom-left shows current map scale")
print()
print("💡 TRY IT: Click the layers icon in the top-right to switch between:")
print("   - Satellite vs Terrain basemap")
print("   - Pre-fire vs Post-fire RGB")
print("   - NBR layers vs classified severity")

## 🏷️ Demo Step 5 — Add Severity Legend Widget

`WidgetControl` lets you embed any ipywidget (HTML, sliders, dropdowns) directly inside the map.

In [ ]:
# ── Step 5: Add an HTML legend using WidgetControl ──────────────────────────
# WidgetControl embeds any ipywidget inside the map at a specified position.

legend_html = """
<div style="
    background: rgba(13,17,23,0.92);
    border: 1px solid #00D4FF;
    border-radius: 6px;
    padding: 10px 14px;
    font-family: Consolas, monospace;
    font-size: 12px;
    color: #E6EDF3;
    min-width: 190px;
">
  <b style="color:#00D4FF;">🔥 Burn Severity (dNBR)</b><br><br>
  <div style="margin:3px 0"><span style="display:inline-block;width:14px;height:14px;background:#006400;border-radius:2px;margin-right:7px;vertical-align:middle;"></span>Enhanced Regrowth</div>
  <div style="margin:3px 0"><span style="display:inline-block;width:14px;height:14px;background:#90EE90;border-radius:2px;margin-right:7px;vertical-align:middle;"></span>Unburned</div>
  <div style="margin:3px 0"><span style="display:inline-block;width:14px;height:14px;background:#FFD700;border-radius:2px;margin-right:7px;vertical-align:middle;"></span>Low Severity</div>
  <div style="margin:3px 0"><span style="display:inline-block;width:14px;height:14px;background:#FF8C00;border-radius:2px;margin-right:7px;vertical-align:middle;"></span>Moderate Severity</div>
  <div style="margin:3px 0"><span style="display:inline-block;width:14px;height:14px;background:#8B0000;border-radius:2px;margin-right:7px;vertical-align:middle;"></span>High Severity</div>
  <hr style="border-color:#21262D;margin:8px 0">
  <span style="color:#8B949E;font-size:10px;">Source: USGS dNBR thresholds</span>
</div>
"""

legend_widget = widgets.HTML(value=legend_html)
legend_control = WidgetControl(widget=legend_widget, position='bottomright')
m.add_control(legend_control)

print("✅ Severity legend added to bottom-right corner of the map")
print()
print("🎉 Full interactive map is ready!")
print("   Scroll up to interact with the map above ↑")

---
## ✨ Bonus — Opacity Slider

We can use ipywidgets to add an interactive slider that controls the opacity of the severity layer — showing how ipyleaflet integrates with the broader widget ecosystem.

In [ ]:
# ── Bonus: opacity slider linked to severity layer ─────────────────────────

opacity_slider = widgets.FloatSlider(
    value=0.85, min=0.0, max=1.0, step=0.05,
    description='Severity opacity:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='400px')
)

def update_opacity(change):
    layer_severity.opacity = change['new']

opacity_slider.observe(update_opacity, names='value')

display(widgets.VBox([
    widgets.Label("🎛️  Drag the slider to adjust the severity layer opacity:"),
    opacity_slider
]))

print("✅ Slider linked to the severity layer — try dragging it!")
print("   This shows how ipywidgets + ipyleaflet create rich interactive UIs.")

---
## 📋 Summary — What We Built

| Component | ipyleaflet Class | Purpose |
|-----------|------------------|---------|
| Interactive canvas | `Map()` | Base widget, sets center/zoom |
| Background tiles | `basemap_to_tiles()` | Satellite / terrain |
| Raster overlays | `ImageOverlay()` | Pre/post RGB, NBR, severity |
| Toggle panel | `LayersControl()` | Show/hide layers |
| Embedded widget | `WidgetControl()` | Legend, sliders |
| Scale bar | `ScaleControl()` | Map scale reference |

---

### 📚 Resources
- **ipyleaflet docs:** https://ipyleaflet.readthedocs.io
- **GitHub:** https://github.com/jupyter-widgets/ipyleaflet
- **USGS Burn Severity:** https://burnseverity.cr.usgs.gov
- **Sentinel-2 bands:** ESA Copernicus Open Access Hub